## Google Colab Setup

Run the cell below once when using Colab. It installs dependencies and checks for GPU.

In [ ]:
import subprocess, sys

# Install dependencies (safe to run on Colab or locally)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "torch", "sentencepiece", "pandas", "numpy", "tqdm"],
    check=False,
)

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("Running on CPU — consider switching to a GPU runtime on Colab.")

# Assignment 4: Prompt Engineering for NER

**Student:** MingHsiang Lee  
**Task:** Named Entity Recognition (NER) with local CoNLL-2003

## 0-shot vs Few-shot

- **0-shot**: instruction + input only, no examples.
- **Few-shot**: instruction + a few labeled examples + input.

This notebook compares both approaches.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

USE_FAST_MODE = True
FAST_SENTENCE_LIMIT = 80
FULL_SENTENCE_LIMIT = None  # Set an integer if you want to cap full mode.

# Local test uses small/base.
# On Colab GPU uncomment large/xl for significantly better F1:
#   - flan-t5-large  (780M): good quality, fits on Colab T4 (15 GB)
#   - flan-t5-xl     (3B):   best quality, needs ~10 GB VRAM
MODEL_NAMES = [
    "google/flan-t5-small",    # 80M  params – fast CPU baseline
    "google/flan-t5-base",     # 250M params
    # "google/flan-t5-large",  # 780M params – recommended on Colab
    # "google/flan-t5-xl",     # 3B   params – best results
]

# Beam search width (higher = better structured JSON output, slower).
NUM_BEAMS = 4

# Relative paths only (no absolute path).
CONLL_REL_PATHS = [
    Path("Assignment2_Name_Entity_Recognition/conll2003"),
    Path("../Assignment2_Name_Entity_Recognition/conll2003"),
]

CONLL_DIR = None
for p in CONLL_REL_PATHS:
    if p.exists():
        CONLL_DIR = p
        break

if CONLL_DIR is None:
    raise FileNotFoundError("Could not find local CoNLL folder via relative paths.")

EVAL_FILE = "eng.testa"
print("Using CoNLL directory:", CONLL_DIR)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

In [ ]:
def read_conll(file_path: Path):
    rows = []
    tokens, tags = [], []
    with file_path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                if tokens:
                    rows.append({"tokens": tokens, "tags": tags})
                    tokens, tags = [], []
                continue
            if line.startswith("-DOCSTART-"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                tokens.append(parts[0])
                tags.append(parts[-1])
    if tokens:
        rows.append({"tokens": tokens, "tags": tags})
    return rows


def bio_to_entities(tokens, tags):
    out = []
    i = 0
    while i < len(tags):
        tag = tags[i]
        if tag == "O" or "-" not in tag:
            i += 1
            continue
        prefix, label = tag.split("-", 1)
        if prefix not in {"B", "I"}:
            i += 1
            continue
        ent = [tokens[i]]
        i += 1
        while i < len(tags) and tags[i] == f"I-{label}":
            ent.append(tokens[i])
            i += 1
        out.append({"entity": " ".join(ent), "type": label})
    return out


sentences = read_conll(CONLL_DIR / EVAL_FILE)
data = []
for s in sentences:
    text = " ".join(s["tokens"])
    gold_entities = bio_to_entities(s["tokens"], s["tags"])
    data.append({"text": text, "gold_entities": gold_entities})

eval_df = pd.DataFrame(data)

if USE_FAST_MODE:
    eval_df = eval_df.head(FAST_SENTENCE_LIMIT).copy()
elif FULL_SENTENCE_LIMIT is not None:
    eval_df = eval_df.head(FULL_SENTENCE_LIMIT).copy()

print("Mode:", "FAST" if USE_FAST_MODE else "FULL")
print("Sentence count:", len(eval_df))
display(eval_df.head(3))

In [ ]:
PROMPTS = {
    # --- zero-shot: plain instruction, no examples ---
    "zero_shot_plain": (
        "Named Entity Recognition task.\n"
        "Extract all named entities from the text.\n"
        "Classify each as: PER (person), ORG (organization), LOC (location), MISC (other named entity).\n"
        "The text may be in ALL CAPS. Still extract entities.\n"
        "Respond with a JSON array only. No explanation. No code.\n"
        'Format: [{"entity": "name", "type": "TYPE"}, ...]\n'
        "If no named entities exist, respond: []\n"
        "\n"
        "Text: {text}\n"
        "JSON:"
    ),
    # --- zero-shot: rubric with explicit type definitions ---
    "zero_shot_rubric": (
        "You are an expert Named Entity Recognition system.\n"
        "Label every named entity in the input text using these rules:\n"
        "  PER  = person name (e.g. Barack Obama, Sampras)\n"
        "  ORG  = organization, team, or company (e.g. Reuters, FIFA, Leicestershire)\n"
        "  LOC  = location or place (e.g. London, France, Africa)\n"
        "  MISC = other named entity (e.g. nationality adjective, event name)\n"
        "The text may be in ALL CAPS. Still identify entities.\n"
        'Output ONLY a JSON array: [{"entity": "...", "type": "..."}]. No code. No prose.\n'
        "\n"
        "Text: {text}\n"
        "JSON:"
    ),
    # --- few-shot: 3 caps-style examples that match CoNLL text format ---
    "few_shot_caps": (
        "Extract named entities. Types: PER, ORG, LOC, MISC. Return JSON array only.\n"
        "\n"
        "Text: LEICESTERSHIRE BEAT NOTTINGHAMSHIRE IN CRICKET FINAL .\n"
        'Output: [{"entity": "LEICESTERSHIRE", "type": "ORG"}, {"entity": "NOTTINGHAMSHIRE", "type": "ORG"}]\n'
        "\n"
        "Text: LONDON 1996-08-30\n"
        'Output: [{"entity": "LONDON", "type": "LOC"}]\n'
        "\n"
        "Text: UNITED STATES BEAT GERMANY 2-1 .\n"
        'Output: [{"entity": "UNITED STATES", "type": "LOC"}, {"entity": "GERMANY", "type": "LOC"}]\n'
        "\n"
        "Text: {text}\n"
        "Output:"
    ),
    # --- few-shot: balanced (includes a no-entity example to reduce false positives) ---
    "few_shot_balanced": (
        "Extract named entities from text. Types: PER=person, ORG=organization, LOC=location, MISC=other.\n"
        "Return JSON array only.\n"
        "\n"
        "Text: THE MATCH WAS PLAYED IN PARIS .\n"
        'Output: [{"entity": "PARIS", "type": "LOC"}]\n'
        "\n"
        "Text: JOHN SMITH JOINS REUTERS IN NEW YORK .\n"
        'Output: [{"entity": "JOHN SMITH", "type": "PER"}, {"entity": "REUTERS", "type": "ORG"}, {"entity": "NEW YORK", "type": "LOC"}]\n'
        "\n"
        "Text: THE WEATHER WAS FINE YESTERDAY .\n"
        "Output: []\n"
        "\n"
        "Text: {text}\n"
        "Output:"
    ),
}


def build_prompt(name, text):
    return PROMPTS[name].replace("{text}", text.strip())

In [ ]:
def build_generator(model_name):
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return {"tokenizer": tokenizer, "model": model, "device": device}


generators = {}
for model_name in MODEL_NAMES:
    print(f"Loading {model_name} ...")
    generators[model_name] = build_generator(model_name)

print("Loaded models:", list(generators.keys()))

In [ ]:
ALLOWED_TYPES = {"PER", "ORG", "LOC", "MISC"}


def parse_ner_output(raw_text):
    text = (raw_text or "").strip()
    left = text.find("[")
    right = text.rfind("]")
    candidate = text[left : right + 1] if left != -1 and right != -1 and right > left else text

    parse_ok = True
    entities = []

    try:
        loaded = json.loads(candidate)
        if isinstance(loaded, dict):
            loaded = [loaded]
        if not isinstance(loaded, list):
            loaded = []

        for item in loaded:
            if not isinstance(item, dict):
                continue
            ent = str(item.get("entity", "")).strip()
            typ = str(item.get("type", "")).strip().upper()
            if ent and typ in ALLOWED_TYPES:
                entities.append({"entity": ent, "type": typ})
    except Exception:
        parse_ok = False

    return entities, parse_ok


def to_entity_set(items):
    out = set()
    for item in items:
        ent = " ".join(str(item.get("entity", "")).lower().split())
        typ = str(item.get("type", "")).upper().strip()
        if ent and typ in ALLOWED_TYPES:
            out.add((ent, typ))
    return out


def generate_text(gen, prompt, max_new_tokens=256):
    tokenizer = gen["tokenizer"]
    model = gen["model"]
    model_device = gen["device"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def evaluate_prompt_model(df, prompt_name, model_name):
    gen = generators[model_name]

    tp = fp = fn = 0
    valid = exact = 0
    rows = []

    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"{model_name} | {prompt_name}"):
        prompt = build_prompt(prompt_name, row.text)
        raw = generate_text(gen, prompt)

        pred_entities, parse_ok = parse_ner_output(raw)
        valid += int(parse_ok)

        gold_set = to_entity_set(row.gold_entities)
        pred_set = to_entity_set(pred_entities)

        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)
        exact += int(gold_set == pred_set)

        rows.append(
            {
                "text": row.text,
                "gold_entities": row.gold_entities,
                "pred_entities": pred_entities,
                "raw_output": raw,
            }
        )

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    metrics = {
        "model": model_name,
        "prompt_template": prompt_name,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "valid_json_rate": round(valid / len(df), 4),
        "exact_sentence_match": round(exact / len(df), 4),
    }

    return metrics, pd.DataFrame(rows)

In [ ]:
all_metrics = []
all_predictions = {}

for model_name in MODEL_NAMES:
    for prompt_name in PROMPTS.keys():
        metrics_row, pred_df = evaluate_prompt_model(eval_df, prompt_name, model_name)
        all_metrics.append(metrics_row)
        all_predictions[(model_name, prompt_name)] = pred_df

metrics_df = (
    pd.DataFrame(all_metrics)
    .sort_values(by=["f1", "precision", "recall"], ascending=False)
    .reset_index(drop=True)
)

display(metrics_df)

In [ ]:
shot_df = metrics_df.copy()
shot_df["shot_type"] = shot_df["prompt_template"].apply(
    lambda x: "few-shot" if x.startswith("few_shot") else "zero-shot"
)
shot_df = (
    shot_df.groupby(["model", "shot_type"], as_index=False)[
        ["precision", "recall", "f1", "valid_json_rate", "exact_sentence_match"]
    ]
    .mean()
    .sort_values(by=["model", "shot_type"])
)

print("0-shot vs few-shot summary:")
display(shot_df)

# Save logs for report evidence
log_examples = eval_df.head(min(5, len(eval_df))).copy()
log_rows = []

for idx, row in log_examples.iterrows():
    for model_name in MODEL_NAMES:
        for prompt_name in PROMPTS.keys():
            prompt = build_prompt(prompt_name, row["text"])
            raw = generate_text(generators[model_name], prompt)
            pred, ok = parse_ner_output(raw)
            log_rows.append(
                {
                    "example_id": idx,
                    "model": model_name,
                    "prompt_template": prompt_name,
                    "parse_ok": ok,
                    "text": row["text"],
                    "gold_entities": row["gold_entities"],
                    "pred_entities": pred,
                    "raw_output": raw,
                }
            )

logs_df = pd.DataFrame(log_rows)
display(logs_df.head(20))
logs_df.to_csv("ner_prompt_output_logs.csv", index=False)
print("Saved logs to ner_prompt_output_logs.csv")

## Bias and Mitigation

Potential bias sources in prompt-based NER:
- Entity overfitting to frequent names/regions in news.
- Prompt-example imbalance by entity type.
- Domain shift from news text to other domains.

Mitigation:
1. Use balanced few-shot examples.
2. Include no-entity examples (`[]`) to reduce false positives.
3. Evaluate per entity type and per domain.
4. Enforce strict output schema and parse checks.

### Deliverable Coverage
- Multiple prompt designs: yes
- 0-shot vs few-shot: yes
- Multi-model comparison: yes
- Output logs: yes
- Bias discussion: yes